In [32]:
from google.colab import drive  # pyright: ignore[reportMissingImports]
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import KFold, cross_val_score
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate
# Format floating-point numbers
pd.options.display.float_format = '{:.2f}'.format
drive.mount('/content/drive', force_remount=True)
PATH = '/content/drive/MyDrive/Applied Statistics/Datasets/'

Mounted at /content/drive


In [3]:
df = pd.read_csv(PATH + 'building_energy.csv')
df

,annual_energy_consumption,building_area,heated_area,cooled_area,building_age,insulation_score,heating_efficiency,occupancy_rate,working_hours,temperature_variability,appliance_density,solar_exposure,maintenance_score
0,32181.40,797.40,733.60,478.40,25,100.00,92.20,0.61,56.40,14.60,1.00,0.42,70.50
1,53253.40,2465.30,2268.10,1479.20,14,95.40,92.70,0.73,67.40,8.10,0.69,0.16,70.30
2,10308.40,1005.90,925.40,603.50,13,100.00,100.00,0.63,27.20,8.00,0.51,0.59,73.00
3,47833.70,1300.50,1196.40,780.30,66,60.30,69.00,0.96,52.30,11.60,0.42,0.41,32.70
4,49504.00,1404.80,1292.40,842.90,71,50.90,78.90,0.88,61.30,15.40,0.66,0.48,43.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,39306.30,657.60,605.00,394.60,64,52.70,74.00,0.49,65.00,13.00,0.12,0.55,56.40
516,20079.10,1254.10,1153.70,752.40,17,96.50,89.20,0.64,54.60,4.90,0.28,0.95,71.20
517,47420.80,1283.00,1180.40,769.80,67,51.30,69.70,0.69,83.00,11.80,0.66,0.67,52.10
518,42832.10,901.90,829.70,541.10,79,60.00,63.20,0.68,92.20,11.00,0.60,0.73,41.90


In [20]:
df.describe()
df.describe().to_excel("summary_2.xlsx")

In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.30, random_state=123)
features = df.select_dtypes(
    include=["int64", "float64"]).columns.tolist()
TARGET = "annual_energy_consumption"
features.remove(TARGET)
features

['building_area',
 'heated_area',
 'cooled_area',
 'building_age',
 'insulation_score',
 'heating_efficiency',
 'occupancy_rate',
 'working_hours',
 'temperature_variability',
 'appliance_density',
 'solar_exposure',
 'maintenance_score']

In [56]:
X = train_df[features]
y = train_df[TARGET]
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), features)])
lasso_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Lasso(max_iter=10000))
])

lasso_params = {
    'regressor__alpha': np.logspace(0, 4, 50)
}
lasso_nested = GridSearchCV(
    lasso_pipeline,
    lasso_params,
    cv=5,
    scoring='r2'
)

outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = (
    'neg_mean_squared_error',
    'r2'
)

lasso_scores = cross_validate(
    lasso_nested,
    X,
    y,
    scoring=scoring,
    cv=outer_cv,
    return_estimator=True
)


def summarize_nested_cv(name, scores):
    mse = -scores['test_neg_mean_squared_error']
    r2 = scores['test_r2']

    print(f"\n{name} Nested Cross-Validation Results:")
    print(
        f"  RMSE : Mean = {np.sqrt(mse).mean():.3f}, SD = {np.sqrt(mse).std():.3f}")
    print(f"  R²  : Mean = {r2.mean():.3f}, SD = {r2.std():.3f}")


summarize_nested_cv("Lasso", lasso_scores)


lasso_alphas = [
    est.best_params_['regressor__alpha']
    for est in lasso_scores['estimator']
]


print("\nLasso selected alphas across outer folds:")
print(lasso_alphas)

final_lasso = GridSearchCV(
    lasso_pipeline,
    lasso_params,
    cv=5,
    scoring='r2'
)

final_lasso.fit(X, y)

best_alpha = final_lasso.best_params_['regressor__alpha']

print("\nSelected alpha by CV")
print(best_alpha)

print("\nLasso R2-index")
print(final_lasso.best_score_)


Lasso Nested Cross-Validation Results:
  RMSE : Mean = 7688.656, SD = 440.112
  R²  : Mean = 0.838, SD = 0.030

Lasso selected alphas across outer folds:
[np.float64(91.02981779915217), np.float64(1.2067926406393286), np.float64(109.85411419875584), np.float64(62.505519252739695), np.float64(109.85411419875584)]

Selected alpha by CV
109.85411419875584

Lasso R2-index
0.8375637867206602


In [ ]:
final_lasso = Lasso(max_iter=10000, alpha = best_alpha)

final_lasso.fit(X, y)
final_lasso.

Lasso(alpha=np.float64(109.85411419875584), max_iter=10000)

In [53]:
# Intercepts
lasso_intercept = final_lasso.intercept_

# Coefficients
lasso_coefs = final_lasso.coef_

coef_df = pd.DataFrame(
    {'Lasso': [lasso_intercept] + list(lasso_coefs)})

coef_df.to_excel("lasso_coeffs.xlsx")